In [1]:
import pandas as pd
import numpy as np
import json

In [4]:
df = pd.read_csv("data/Indian ODI Cricketers.csv")
df.head()

,No,Name,First,Last,Mat,Inn,NO.1,Runs,HS,Avg,Balls,Mdn,Runs_1,Wkt,BBM,Avg_2,Ca,St
0,1,Syed Abid Ali,1974,1975,5,3,0,93,70,31.00,336,10,187,7,2/22,26.71,0,0
1,2,Bishan Singh Bedi,1974,1979,10,7,2,31,13,6.20,590,17,340,7,2/44,48.57,4,0
2,3,Farokh Engineer,1974,1975,5,4,1,114,54*,38.00,—,—,—,—,—,—,3,1
3,4,Sunil Gavaskar,1974,1987,108,102,14,3092,103*,35.13,20,0,25,1,1/10,25.00,22,0
4,5,Madan Lal,1974,1987,67,35,14,401,53*,19.09,3164,44,2137,73,4/20,29.27,18,0


In [5]:
df.fillna(0, inplace=True)

numeric_cols = ["Matches", "Runs", "Average", "StrikeRate", "Wickets"]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

df.info()

TypeError: Invalid value '0' for dtype 'str'. Value should be a string or missing value, got 'int' instead.

In [6]:
df.rename(columns={
    "Name": "Player",
    "Mat": "Matches",
    "Runs": "Runs",
    "Avg": "Bat_Avg",
    "Wkt": "Wickets",
    "Avg_2": "Bowl_Avg",
    "Ca": "Catches",
    "St": "Stumpings"
}, inplace=True)

In [7]:
cols = ["Matches", "Runs", "Bat_Avg", "Wickets", "Bowl_Avg", "Catches", "Stumpings"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

In [8]:
def calculate_score(row):
    score = 0

    # Batting weight
    score += row["Runs"] * 0.25
    score += row["Bat_Avg"] * 0.4

    # Bowling weight
    score += row["Wickets"] * 0.5
    
    # Bowling average (lower is better → invert)
    if row["Bowl_Avg"] > 0:
        score += (50 - row["Bowl_Avg"]) * 0.3

    # Experience
    score += row["Matches"] * 0.1

    # Fielding bonus
    score += (row["Catches"] + row["Stumpings"]) * 0.1

    return score

In [9]:
df["score"] = df.apply(calculate_score, axis=1)

In [10]:
top_11 = df.sort_values(by="score", ascending=False).head(11)

top_11[["Player", "score"]]

,Player,score
73,Sachin Tendulkar,4763.388
174,Virat Kohli,3256.053
83,Sourav Ganguly,2915.825
94,Rahul Dravid,2766.910
156,M.S. Dhoni,2754.542
167,Rohit Sharma,2508.391
50,Mohammad Azharuddin,2417.295
133,Yuvraj Singh,2264.712
122,Virender Sehwag,2096.073
187,Shikhar Dhawan,1740.894


In [11]:
captain = top_11.iloc[0]["Player"]

In [12]:
result = {
    "squad": top_11["Player"].tolist(),
    "captain": captain,
    "explanation": "Selected based on batting, bowling, and fielding performance"
}

result

{'squad': ['Sachin Tendulkar ',
  'Virat Kohli ',
  'Sourav Ganguly ',
  'Rahul Dravid ',
  'M.S. Dhoni ',
  'Rohit Sharma ',
  'Mohammad Azharuddin ',
  'Yuvraj Singh',
  'Virender Sehwag ',
  'Shikhar Dhawan ',
  'Suresh Raina '],
 'captain': 'Sachin Tendulkar ',
 'explanation': 'Selected based on batting, bowling, and fielding performance'}

In [13]:
df['batsman_score'] = (
    df['Batting_Average'] * 0.4 +
    df['Strike_Rate'] * 0.3 +
    df['Runs'] * 0.2 +
    df['Hundreds'] * 10 +
    df['Fifties'] * 5
)

KeyError: 'Batting_Average'

In [14]:
print(df.columns)

Index(['No', 'Player', 'First', 'Last', 'Matches', 'Inn', 'NO.1', 'Runs', 'HS',
       'Bat_Avg', 'Balls', 'Mdn', 'Runs_1', 'Wickets', 'BBM', 'Bowl_Avg',
       'Catches', 'Stumpings', 'score'],
      dtype='str')


In [15]:
df['batsman_score'] = ( df['Bat_Avg'] * 0.4 + df['Strike_Rate'] * 0.3 + df['Runs'] * 0.2 + df['Hundreds'] * 10 + df['Fifties'] * 5 )

KeyError: 'Strike_Rate'

In [16]:
df.fillna(0, inplace=True)

# Avoid division issues
df['Bat_Avg'].replace(0, 1, inplace=True)
df['Bowl_Avg'].replace(0, 1, inplace=True)
df['Balls'].replace(0, 1, inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_17368\3018698958.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Bat_Avg'].replace(0, 1, inplace=True)
C:\Users\User\AppData\Local\Temp\ipykernel_17368\3018698958.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an

0       336
1       590
2         —
3        20
4      3164
       ... 
245      48
246     156
247      79
248     330
249      30
Name: Balls, Length: 250, dtype: str

In [17]:
df['strike_rate'] = (df['Runs'] / df['Balls']) * 100

TypeError: unsupported operand type(s) for /: 'float' and 'str'

In [18]:
df['Runs'] = pd.to_numeric(df['Runs'], errors='coerce')
df['Balls'] = pd.to_numeric(df['Balls'], errors='coerce')
df['Bat_Avg'] = pd.to_numeric(df['Bat_Avg'], errors='coerce')
df['Wickets'] = pd.to_numeric(df['Wickets'], errors='coerce')
df['Bowl_Avg'] = pd.to_numeric(df['Bowl_Avg'], errors='coerce')
df['Mdn'] = pd.to_numeric(df['Mdn'], errors='coerce')

In [19]:
df.fillna(0, inplace=True)

,No,Player,First,Last,Matches,Inn,NO.1,Runs,HS,Bat_Avg,Balls,Mdn,Runs_1,Wickets,BBM,Bowl_Avg,Catches,Stumpings,score
0,1,Syed Abid Ali,1974,1975,5,3,0,93.0,70,31.00,336.0,10.0,187,7.0,2/22,26.71,0,0,46.637
1,2,Bishan Singh Bedi,1974,1979,10,7,2,31.0,13,6.20,590.0,17.0,340,7.0,2/44,48.57,4,0,15.559
2,3,Farokh Engineer,1974,1975,5,4,1,114.0,54*,38.00,0.0,0.0,—,0.0,—,0.00,3,1,44.600
3,4,Sunil Gavaskar,1974,1987,108,102,14,3092.0,103*,35.13,20.0,0.0,25,1.0,1/10,25.00,22,0,808.052
4,5,Madan Lal,1974,1987,67,35,14,401.0,53*,19.09,3164.0,44.0,2137,73.0,4/20,29.27,18,0,159.105
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,Ravi Bishnoi,2022,2022,1,1,1,4.0,4*,0.00,48.0,0.0,69,1.0,1/69,69.00,0,0,-4.100
246,247,Shahbaz Ahmed,2022,2022,3,0,0,0.0,0,0.00,156.0,0.0,125,3.0,2/32,41.66,1,0,4.402
247,248,Arshdeep Singh,2022,2022,3,1,0,9.0,9,9.00,79.0,1.0,89,0.0,—,0.00,0,0,6.150
248,249,Umran Malik,2022,2023,8,3,3,2.0,2*,0.00,330.0,2.0,355,13.0,3/57,27.30,1,0,14.710


In [20]:
df['Balls'].replace(0, 1, inplace=True)

C:\Users\User\AppData\Local\Temp\ipykernel_17368\1162386348.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Balls'].replace(0, 1, inplace=True)


0       336.0
1       590.0
2         1.0
3        20.0
4      3164.0
        ...  
245      48.0
246     156.0
247      79.0
248     330.0
249      30.0
Name: Balls, Length: 250, dtype: float64

In [21]:
df['Balls'] = df['Balls'].replace(0, 1)

In [22]:
df['strike_rate'] = (df['Runs'] / df['Balls']) * 100

In [23]:
numeric_cols = ['Runs', 'Balls', 'Bat_Avg', 'Wickets', 'Bowl_Avg', 'Mdn']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')
df.fillna(0, inplace=True)

,No,Player,First,Last,Matches,Inn,NO.1,Runs,HS,Bat_Avg,Balls,Mdn,Runs_1,Wickets,BBM,Bowl_Avg,Catches,Stumpings,score,strike_rate
0,1,Syed Abid Ali,1974,1975,5,3,0,93.0,70,31.00,336.0,10.0,187,7.0,2/22,26.71,0,0,46.637,27.678571
1,2,Bishan Singh Bedi,1974,1979,10,7,2,31.0,13,6.20,590.0,17.0,340,7.0,2/44,48.57,4,0,15.559,5.254237
2,3,Farokh Engineer,1974,1975,5,4,1,114.0,54*,38.00,1.0,0.0,—,0.0,—,0.00,3,1,44.600,11400.000000
3,4,Sunil Gavaskar,1974,1987,108,102,14,3092.0,103*,35.13,20.0,0.0,25,1.0,1/10,25.00,22,0,808.052,15460.000000
4,5,Madan Lal,1974,1987,67,35,14,401.0,53*,19.09,3164.0,44.0,2137,73.0,4/20,29.27,18,0,159.105,12.673831
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,Ravi Bishnoi,2022,2022,1,1,1,4.0,4*,0.00,48.0,0.0,69,1.0,1/69,69.00,0,0,-4.100,8.333333
246,247,Shahbaz Ahmed,2022,2022,3,0,0,0.0,0,0.00,156.0,0.0,125,3.0,2/32,41.66,1,0,4.402,0.000000
247,248,Arshdeep Singh,2022,2022,3,1,0,9.0,9,9.00,79.0,1.0,89,0.0,—,0.00,0,0,6.150,11.392405
248,249,Umran Malik,2022,2023,8,3,3,2.0,2*,0.00,330.0,2.0,355,13.0,3/57,27.30,1,0,14.710,0.606061


In [24]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

features = ['Bat_Avg', 'strike_rate', 'Runs', 'Wickets', 'Bowl_Avg']

df[features] = scaler.fit_transform(df[features])

In [25]:
df['batsman_score'] = (
    df['Bat_Avg'] * 0.35 +
    df['strike_rate'] * 0.25 +
    df['Runs'] * 0.20 +
    df['Hundreds'] * 0.10 +
    df['Fifties'] * 0.10
)

KeyError: 'Hundreds'

In [26]:
df['batsman_score'] = (
    df['Bat_Avg'] * 0.4 +
    df['strike_rate'] * 0.3 +
    df['Runs'] * 0.2 +
    df['Matches'] * 0.1
)

In [27]:
df['consistency'] = df['Runs'] / df['Inn']

TypeError: unsupported operand type(s) for /: 'float' and 'str'

In [28]:
numeric_cols = ['Runs', 'Inn', 'Balls', 'Bat_Avg', 'Wickets', 'Bowl_Avg', 'Matches', 'Mdn']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [29]:
df.fillna(0, inplace=True)

,No,Player,First,Last,Matches,Inn,NO.1,Runs,HS,Bat_Avg,...,Mdn,Runs_1,Wickets,BBM,Bowl_Avg,Catches,Stumpings,score,strike_rate,batsman_score
0,1,Syed Abid Ali,1974,1975,5,3.0,0,0.005047,70,0.469697,...,10.0,187,0.020772,2/22,0.134221,0,0,46.637,4.074573e-05,0.688900
1,2,Bishan Singh Bedi,1974,1979,10,7.0,2,0.001682,13,0.093939,...,17.0,340,0.020772,2/44,0.244070,4,0,15.559,7.734782e-06,1.037915
2,3,Farokh Engineer,1974,1975,5,4.0,1,0.006187,54*,0.575758,...,0.0,—,0.000000,—,0.000000,3,1,44.600,1.678198e-02,0.736575
3,4,Sunil Gavaskar,1974,1987,108,102.0,14,0.167806,103*,0.532273,...,0.0,25,0.002967,1/10,0.125628,22,0,808.052,2.275872e-02,11.053298
4,5,Madan Lal,1974,1987,67,35.0,14,0.021763,53*,0.289242,...,44.0,2137,0.216617,4/20,0.147085,18,0,159.105,1.865719e-05,6.820055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,246,Ravi Bishnoi,2022,2022,1,1.0,1,0.000217,4*,0.000000,...,0.0,69,0.002967,1/69,0.346734,0,0,-4.100,1.226753e-05,0.100047
246,247,Shahbaz Ahmed,2022,2022,3,0.0,0,0.000000,0,0.000000,...,0.0,125,0.008902,2/32,0.209347,1,0,4.402,0.000000e+00,0.300000
247,248,Arshdeep Singh,2022,2022,3,1.0,0,0.000488,9,0.136364,...,1.0,89,0.000000,—,0.000000,0,0,6.150,1.677080e-05,0.354648
248,249,Umran Malik,2022,2023,8,3.0,3,0.000109,2*,0.000000,...,2.0,355,0.038576,3/57,0.137186,1,0,14.710,8.921840e-07,0.800022


In [30]:
df['Inn'] = df['Inn'].replace(0, 1)
df['Balls'] = df['Balls'].replace(0, 1)
df['Bowl_Avg'] = df['Bowl_Avg'].replace(0, 1)

In [31]:
df['consistency'] = df['Runs'] / df['Inn']

In [32]:
df['batsman_score'] = (
    df['Bat_Avg'] * 0.35 +
    df['strike_rate'] * 0.25 +
    df['Runs'] * 0.20 +
    df['consistency'] * 0.20
)

In [33]:
top_batsmen = df.sort_values(by='batsman_score', ascending=False).head(11)

print(top_batsmen[['Player', 'Runs', 'Bat_Avg', 'strike_rate', 'batsman_score']])

                Player      Runs   Bat_Avg  strike_rate  batsman_score
187    Shikhar Dhawan   0.368664  0.668333     1.000000       0.558099
174       Virat Kohli   0.699989  0.868485     0.002962       0.445236
73   Sachin Tendulkar   1.000000  0.679242     0.000337       0.438262
226       Shubman Gill  0.071149  0.993182     0.192993       0.410685
156        M.S. Dhoni   0.575220  0.761061     0.043341       0.392642
240       Sanju Samson  0.017909  1.000000     0.048579       0.366085
167      Rohit Sharma   0.533214  0.736818     0.002439       0.365591
232      Krunal Pandya  0.007055  0.984848     0.000084       0.346482
83     Sourav Ganguly   0.608976  0.620455     0.000364       0.339455
212         KL Rahul    0.107782  0.683788     0.292360       0.334387
190    Ajinkya Rahane   0.160751  0.534242     0.436037       0.328514


In [34]:
df['bowler_score'] = (
    df['Wickets'] * 0.4 +
    (1 / df['Bowl_Avg']) * 0.3 +   # lower avg = better
    df['Mdn'] * 0.2 +
    df['Matches'] * 0.1
)

In [35]:
top_bowlers = df.sort_values(by='bowler_score', ascending=False).head(11)

print(top_bowlers[['Player', 'Wickets', 'Bowl_Avg', 'bowler_score']])

                Player   Wickets  Bowl_Avg  bowler_score
24          Kapil Dev   0.750742  0.137940     71.975160
80     Javagal Srinath  0.934718  0.141106     52.799956
73   Sachin Tendulkar   0.456973  0.223518     52.624966
77        Anil Kumble   1.000000  0.153568     51.053534
132        Zaheer Khan  0.798220  0.151307     44.102018
112    Harbhajan Singh  0.786350  0.168191     42.098227
110       Ajit Agarkar  0.854599  0.139950     41.585466
83     Sourav Ganguly   0.296736  0.192714     38.475409
156        M.S. Dhoni   0.002967  0.155779     36.626993
94       Rahul Dravid   0.011869  0.213568     35.609454
133       Yuvraj Singh  0.326409  0.193065     35.384442


In [36]:
top_batsmen.to_csv("top_batsmen.csv", index=False)
top_bowlers.to_csv("top_bowlers.csv", index=False)

In [37]:
bowlers = df[(df['Wickets'] > 30) & (df['Matches'] > 20)]

In [38]:
bowlers = bowlers.copy()

bowlers['bowler_score'] = (
    bowlers['Wickets'] * 0.5 +
    (1 / bowlers['Bowl_Avg']) * 0.3 +
    bowlers['Mdn'] * 0.2
)

In [39]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

bowlers[['Wickets', 'Bowl_Avg', 'Mdn']] = scaler.fit_transform(
    bowlers[['Wickets', 'Bowl_Avg', 'Mdn']]
)

ValueError: Found array with 0 sample(s) (shape=(0, 3)) while a minimum of 1 is required by MinMaxScaler.

In [40]:
print(len(df))
print(len(bowlers))

250
0


In [41]:
print(df['Wickets'].describe())

count    250.000000
mean       0.078576
std        0.167886
min        0.000000
25%        0.000000
50%        0.008902
75%        0.059347
max        1.000000
Name: Wickets, dtype: float64


In [42]:
bowlers = df[df['Wickets'] > 50]   # raw wickets

In [43]:
bowlers = df[df['Wickets'] > 0.3]

In [44]:
bowlers = df.copy()

In [45]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

bowlers[['Wickets', 'Bowl_Avg', 'Mdn']] = scaler.fit_transform(
    bowlers[['Wickets', 'Bowl_Avg', 'Mdn']]
)

In [46]:
bowlers['bowler_score'] = (
    bowlers['Wickets'] * 0.5 +
    (1 - bowlers['Bowl_Avg']) * 0.3 +
    bowlers['Mdn'] * 0.2
)

In [47]:
top_bowlers = bowlers.sort_values(by='bowler_score', ascending=False).head(11)
print(top_bowlers[['Player', 'Wickets', 'Bowl_Avg', 'bowler_score']])

               Player   Wickets  Bowl_Avg  bowler_score
77       Anil Kumble   1.000000  0.122708      0.855953
80    Javagal Srinath  0.934718  0.109792      0.851017
24         Kapil Dev   0.750742  0.106510      0.843418
110      Ajit Agarkar  0.854599  0.108594      0.779828
132       Zaheer Khan  0.798220  0.120365      0.758320
112   Harbhajan Singh  0.786350  0.137865      0.722454
88   Venkatesh Prasad  0.581602  0.131771      0.618504
176   Ravindra Jadeja  0.566766  0.158281      0.578452
46    Manoj Prabhakar  0.465875  0.113906      0.563447
152      Irfan Pathan  0.513353  0.118333      0.562028
194    Mohammed Shami  0.480712  0.098854      0.548998


In [48]:
batsmen = df[df['Runs'] > 1000]

In [49]:
batsmen

,No,Player,First,Last,Matches,Inn,NO.1,Runs,HS,Bat_Avg,...,Wickets,BBM,Bowl_Avg,Catches,Stumpings,score,strike_rate,batsman_score,consistency,bowler_score
